# 30 · Candidate Generation — Pooling & Audit

**Purpose.** Assemble stage one of the two-stage architecture. Every model so
far competed *solo* on a 10-slot list; from here they stop being competitors
and become **candidate sources**: each contributes a shortlist, the union
(deduplicated, with source flags) becomes the candidate pool, and the XGBoost
ranker (notebook 32) decides the final 10. This notebook builds that pool and
audits it.

## Theory — why two stages, and why recall is the only metric here

The ranker can only rank what the pool contains: a label item missing from
the candidate set is an *irrecoverable* miss, no matter how good the ranker.
So stage one is judged purely on **candidate recall at budget** — the share
of next-30-day purchases present anywhere in the pool — not on ordering
(that's stage two's job) and not @10 (the pool is ~200 items, not 10).
Pooled candidate recall is therefore the **ceiling the ranker inherits**.

The budget is an economic choice: ~200 candidates × ~2,500 households ≈ 500K
rows — comfortable XGBoost training scale — versus scoring the full 90K
catalog per household (~225M rows), which recall-stage filtering exists to
avoid. (Our catalog is small enough that full scoring is *feasible*; the
two-stage discipline is kept because it's the production-realistic pattern.)

## The cast (roles assigned by notebooks 10–21)

| Source | Slice it serves | Budget |
|---|---|---|
| **buy-again** — all previously bought, recency-ranked | repeat (~47% of label pairs, most of the value) | top 150 |
| **global popularity** — staples fallback | discovery + cold-start | top 50 |
| **ALS-new** (log1p, never-bought only) — learned discovery | discovery (53% of label pairs, hardest) | top 50 |

Covisitation-kNN sits out per notebook 20; it re-enters only if this audit
shows a discovery-recall gap ALS can't close.

## Goals

1. **Pooled candidate recall** at budget — total, repeat slice, discovery
   slice. This fixes the ranker's ceiling.
2. **Per-source audit** — solo recall at budget, *unique* contribution
   (label hits no other source provides), and overlap. Sources that
   contribute nothing unique get cut; budgets get rebalanced toward the
   binding slice.
3. **Freeze the candidate config** (sources + budgets) into `configs/`, so
   notebook 31 (features) and 32 (ranker) build on a fixed pool. Source
   membership flags are retained — they become ranker features.

In [9]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import scipy.sparse as sp
from pathlib import Path

from implicit.als import AlternatingLeastSquares
from retail_ds.evaluate.metrics import recall_at_k, hit_rate_at_k, ndcg_at_k

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parents[1]

CFG = yaml.safe_load((ROOT / "configs" / "base.yaml").read_text())
con = duckdb.connect((ROOT / "db" / "retail.duckdb").as_posix(), read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4.5)

AS_OF = CFG["snapshots"]["valid"]   # 600 — audit on validation; ranker will train on 450/510/570
K = CFG["top_k"]

BUDGETS = {k: (10**9 if v == -1 else v) for k, v in CFG["candidates"].items()}
labels = q(f"SELECT household_key, product_id FROM purchase_labels({AS_OF}, 30)")
labels_new = q(f"""
    SELECT l.household_key, l.product_id
    FROM purchase_labels({AS_OF}, 30) l
    LEFT JOIN household_product_snapshot({AS_OF}) h
      USING (household_key, product_id)
    WHERE h.product_id IS NULL
""")
labels_repeat = labels.merge(labels_new, on=["household_key", "product_id"],
                             how="left", indicator=True)
labels_repeat = labels_repeat[labels_repeat["_merge"] == "left_only"].drop(columns="_merge")

print("as_of:", AS_OF, "| budgets:", BUDGETS)
print(f"labels: {len(labels):,} total = {len(labels_repeat):,} repeat + {len(labels_new):,} discovery")

as_of: 600 | budgets: {'buy_again': 1000000000, 'popularity': 50, 'als_new': 50}
labels: 106,669 total = 49,959 repeat + 56,710 discovery


In [10]:
# 1) buy-again: recency-ranked own history, top 150
buy_again = q(f"""
    SELECT household_key, product_id,
           ROW_NUMBER() OVER (
               PARTITION BY household_key
               ORDER BY days_since_last ASC, times_bought DESC
           ) AS rank
    FROM household_product_snapshot({AS_OF})
    QUALIFY rank <= {BUDGETS['buy_again']}
""")

# 2) global popularity: top 50, same list for everyone
popularity = q(f"""
    WITH top_products AS (
        SELECT product_id, COUNT(*) AS n
        FROM staging.stg_transactions
        WHERE day_no <= {AS_OF} AND day_no > {AS_OF} - 365
        GROUP BY product_id ORDER BY n DESC LIMIT {BUDGETS['popularity']}
    )
    SELECT h.household_key, t.product_id
    FROM (SELECT DISTINCT household_key FROM customer_snapshot({AS_OF})) h
    CROSS JOIN top_products t
""")

# 3) ALS-new: log1p ALS, never-bought items only, top 50
interactions = q(f"""
    SELECT household_key, product_id, COUNT(DISTINCT basket_id) AS times_bought
    FROM staging.stg_transactions
    WHERE day_no <= {AS_OF}
    GROUP BY household_key, product_id
""")
households = np.sort(interactions["household_key"].unique())
products   = np.sort(interactions["product_id"].unique())
matrix = sp.csr_matrix(
    (np.log1p(interactions["times_bought"]).astype(np.float32),
     (interactions["household_key"].map({h: i for i, h in enumerate(households)}),
      interactions["product_id"].map({p: i for i, p in enumerate(products)}))),
    shape=(len(households), len(products)))

als = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=20.0,
                              iterations=20, random_state=42)
als.fit(matrix)
ids, _ = als.recommend(np.arange(len(households)), matrix,
                       N=BUDGETS["als_new"], filter_already_liked_items=True)
als_new = pd.DataFrame({
    "household_key": np.repeat(households, BUDGETS["als_new"]),
    "product_id":    products[ids.ravel()],
})
print(f"buy_again {len(buy_again):,} | popularity {len(popularity):,} | als_new {len(als_new):,} rows")

100%|██████████| 20/20 [00:04<00:00,  4.64it/s]


buy_again 1,194,524 | popularity 124,900 | als_new 124,900 rows


In [11]:
pool = pd.concat([
    buy_again[["household_key", "product_id"]].assign(source="buy_again"),
    popularity.assign(source="popularity"),
    als_new.assign(source="als_new"),
])
candidates = (pool.pivot_table(index=["household_key", "product_id"],
                               columns="source", aggfunc="size", fill_value=0)
              .astype(bool).reset_index())

per_household = candidates.groupby("household_key").size()
print(f"pool: {len(candidates):,} candidate rows | "
      f"avg {per_household.mean():.0f} / median {per_household.median():.0f} per household")

pool: 1,358,184 candidate rows | avg 544 / median 452 per household


In [12]:
def cand_recall(cands: pd.DataFrame, labs: pd.DataFrame) -> float:
    hits = labs.merge(cands[["household_key", "product_id"]].drop_duplicates(),
                      on=["household_key", "product_id"])
    return len(hits) / len(labs)

audit = []
for slice_name, labs in [("total", labels), ("repeat", labels_repeat), ("discovery", labels_new)]:
    row = {"slice": slice_name, "pooled": cand_recall(candidates, labs)}
    for source in BUDGETS:
        row[source] = cand_recall(pool[pool["source"] == source], labs)
    audit.append(row)
audit_df = pd.DataFrame(audit).set_index("slice").round(3)

# unique contribution: label hits that ONLY one source provides
labeled = labels.merge(candidates, on=["household_key", "product_id"])
labeled["n_sources"] = labeled[list(BUDGETS)].sum(axis=1)
unique = {s: ((labeled[s]) & (labeled["n_sources"] == 1)).mean() for s in BUDGETS}
print("share of covered labels provided by ONLY this source:",
      {k: round(v, 3) for k, v in unique.items()})
audit_df

share of covered labels provided by ONLY this source: {'buy_again': np.float64(0.813), 'popularity': np.float64(0.01), 'als_new': np.float64(0.021)}


,pooled,buy_again,popularity,als_new
slice,,,,
total,0.489,0.468,0.081,0.016
repeat,1.000,1.000,0.152,0.000
discovery,0.038,0.000,0.019,0.029


**Findings — candidate audit (as-of 600, ~211/household initial).**
Pooled candidate recall 0.300 total / 0.597 repeat / 0.038 discovery.
Headline: buy-again@150 was the binding constraint — recency truncation
dropped {X}% of repeat labels; budget sensitivity: 150 → 0.556, 300 → {X},
unlimited → 1.000 repeat coverage at {X} rows/household. Unique contribution:
buy_again 69.6% / popularity 8.1% / als_new 3.4% — all three retained,
covisitation permanently cut. Discovery is 96% uncoverable at current
generators — accepted, per strategy.
**Frozen config:** buy_again unlimited (~{X}/hh) ∪ popularity 50 ∪ als_new 50;
source flags retained as ranker features. Ranker ceiling: pool contains
0.300 of all labels vs 0.072 captured by the best solo top-10 — a 4×
conversion headroom for notebook 32.

In [13]:
for budget in [150, 300, 1000000]:   # last = effectively unlimited
    ba = q(f"""
        SELECT household_key, product_id,
               ROW_NUMBER() OVER (
                   PARTITION BY household_key
                   ORDER BY days_since_last ASC, times_bought DESC
               ) AS rank
        FROM household_product_snapshot({AS_OF})
        QUALIFY rank <= {budget}
    """)
    print(f"buy_again budget {budget:>7}: repeat coverage {cand_recall(ba, labels_repeat):.3f}"
          f" | rows {len(ba):,}")

buy_again budget     150: repeat coverage 0.556 | rows 340,954
buy_again budget     300: repeat coverage 0.742 | rows 604,420
buy_again budget 1000000: repeat coverage 1.000 | rows 1,194,524


**Findings — candidate audit (as-of 600, frozen config).**
Budget sensitivity exposed buy-again@150 as the binding constraint (recency
truncation dropped 44% of repeat labels: 150 → 0.556, 300 → 0.742,
unlimited → 1.000 at ~478 rows/household). Frozen pool: buy_again unlimited
∪ popularity 50 ∪ als_new 50 → 1.36M candidate rows, avg 544/household.
Pooled candidate recall: **0.489 total / 1.000 repeat / 0.038 discovery**.
Unique contributions kept all three sources; covisitation permanently cut.

**The ranker's inherited arithmetic:** the pool contains 48.9% of all label
pairs; the best solo top-10 (buy-again) captures 7.2%. Converting pool
membership into top-10 placement — a ~7× headroom — is now entirely a
ranking problem. Discovery beyond 3.8% is unreachable at current generators
and deliberately deprioritized.